# Notebook 11: Train / Validation / Test Split

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated time:** 2.5 hours  
**Series:** Week 4 — Data Fundamentals & Preprocessing Pipelines  
**Theme:** Credit Card Fraud Detection

---

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why we need separate train, validation, and test sets — using the exam analogy.
2. Perform stratified splits that preserve class proportions on imbalanced fraud data.
3. Use cross-validation (`StratifiedKFold`) as a more stable alternative to a single holdout.
4. Implement temporal and group-aware splits for realistic fraud scenarios.
5. Articulate the golden rule: **the test set is touched exactly once**.

---

## The Real-World Analogy First

Imagine you are studying for a professional certification exam:

| What you do | ML equivalent |
|---|---|
| Read the textbook and practice problems | **Training set** — the model learns patterns here |
| Take mock / practice exams | **Validation set** — you tune your study strategy based on mock results |
| Sit the real exam on exam day | **Test set** — the single unbiased measure of your true ability |

Now, what if you secretly memorised the answers to the real exam questions the night before?  
You would score 100 % on exam day — but that score is a **lie**. It tells you nothing about whether you actually learned the material.

**Peeking at the test set during model development is exactly the same mistake.**  
Every decision you make after seeing the test set (which model to use, which threshold to set, which features to engineer) uses information from the test set. Your reported accuracy is no longer an unbiased estimate — it is optimistically inflated.

> **The Golden Rule: touch the test set EXACTLY ONCE — at the very end, after all decisions are finalised.**

## Section 0: Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_val_score,
    TimeSeriesSplit,
    GroupKFold,
)
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
print('Libraries loaded successfully.')

## Section 1: Create Synthetic Credit Card Fraud Dataset

We generate **10,000 transactions** spanning 90 days with:
- A **2 % fraud rate** (200 fraudulent transactions out of 10,000) — realistic and highly imbalanced.
- A **timestamp** column — needed for temporal splits.
- A **user_id** column — needed for group-aware splits.
- Numeric features correlated with fraud (higher amounts, unusual hours, etc.).

In [ ]:
np.random.seed(RANDOM_STATE)

N = 10_000
FRAUD_RATE = 0.02
N_FRAUD = int(N * FRAUD_RATE)          # 200 fraud transactions
N_LEGIT = N - N_FRAUD                  # 9,800 legitimate
N_USERS = 1_000                        # 1,000 unique cardholders

# ---- User IDs: each user has ~10 transactions on average -----------------
user_ids = np.random.choice(np.arange(N_USERS), size=N, replace=True)

# ---- Timestamps: uniform spread over 90-day window ----------------------
base_date = datetime(2024, 1, 1)
seconds_in_90_days = 90 * 24 * 3600
timestamps = [base_date + timedelta(seconds=int(s))
              for s in np.sort(np.random.randint(0, seconds_in_90_days, N))]

# ---- Labels --------------------------------------------------------------
is_fraud = np.zeros(N, dtype=int)
# Scatter fraud somewhat later in the window (more realistic)
fraud_indices = np.random.choice(np.arange(N // 2, N), size=N_FRAUD, replace=False)
is_fraud[fraud_indices] = 1

# ---- Features ------------------------------------------------------------
# Legitimate transactions
amount      = np.random.exponential(scale=80, size=N).clip(1, 5000)
hour        = np.random.choice(np.arange(24), size=N,
                               p=np.array([0.5 if h < 6 or h > 22 else 3.0
                                           for h in range(24)]) /
                                 sum(0.5 if h < 6 or h > 22 else 3.0 for h in range(24)))
num_tx_7d   = np.random.poisson(lam=5, size=N).clip(0, 50)
credit_score = np.random.normal(loc=680, scale=80, size=N).clip(300, 850)

# Boost fraud signal: fraud has higher amounts, odd hours, more recent rapid-fire transactions
amount[is_fraud == 1]       += np.random.exponential(scale=300, size=N_FRAUD)
hour[is_fraud == 1]          = np.random.choice([0,1,2,3,22,23], size=N_FRAUD)
num_tx_7d[is_fraud == 1]    += np.random.poisson(lam=10, size=N_FRAUD)
credit_score[is_fraud == 1] -= np.random.uniform(20, 80, size=N_FRAUD)
credit_score = credit_score.clip(300, 850)

df = pd.DataFrame({
    'timestamp'    : timestamps,
    'user_id'      : user_ids,
    'amount'       : amount.round(2),
    'hour'         : hour,
    'num_tx_7d'    : num_tx_7d,
    'credit_score' : credit_score.round(1),
    'is_fraud'     : is_fraud,
})

print(f'Dataset shape : {df.shape}')
print(f'Fraud count   : {df["is_fraud"].sum()} ({df["is_fraud"].mean()*100:.1f} %)')
print(f'Date range    : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Unique users  : {df["user_id"].nunique()}')
df.head()

## Section 2: Two-Way vs Three-Way Split

### When to use each:

| Split type | When to use | Example |
|---|---|---|
| **Train / Test** (two-way) | You have a fixed model, no hyperparameter tuning | Quick baseline experiments |
| **Train / Val / Test** (three-way) | You are selecting models or tuning hyperparameters | Comparing 5 algorithms, choosing C for logistic regression |

### Why separate validation from test?

Suppose you train 10 models and pick the one with the highest **test** AUC.  
By choosing based on test performance, you have implicitly used test information to make a decision.  
The "winning" model's test AUC is no longer a clean estimate — it benefits from selection bias.  
The validation set absorbs this bias; the test set remains untouched.

### Typical ratios

| Dataset size | Recommended ratio |
|---|---|
| < 5,000 rows | Use cross-validation (no held-out val set needed) |
| 5,000–100,000 | 70 / 15 / 15 or 80 / 10 / 10 |
| > 100,000 | 90 / 5 / 5 (absolute counts matter more than percentages) |

In [ ]:
# -----------------------------------------------------------------------
# Three-way stratified split: 70% train, 15% val, 15% test
# -----------------------------------------------------------------------
FEATURES = ['amount', 'hour', 'num_tx_7d', 'credit_score']
TARGET   = 'is_fraud'

X = df[FEATURES]
y = df[TARGET]

# Step 1: split off 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)

# Step 2: from the remaining 85%, split off 15/85 ≈ 17.6% → final 15% val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=RANDOM_STATE
)

print('=== Three-way split ===')
print(f'Train : {len(X_train):>6,} rows  | fraud rate = {y_train.mean()*100:.2f} %')
print(f'Val   : {len(X_val):>6,} rows  | fraud rate = {y_val.mean()*100:.2f} %')
print(f'Test  : {len(X_test):>6,} rows  | fraud rate = {y_test.mean()*100:.2f} %')
print()
expected_pct = y.mean()*100
print(f'Expected fraud rate (overall): {expected_pct:.2f} %')

## Section 3: Stratification — Why It Matters for Fraud Data

Our fraud rate is 2 %. With 10,000 transactions that gives us 200 fraud cases.  
A naive random split can easily place **zero** fraud cases in the test set by chance, especially with smaller datasets.

**`stratify=y`** ensures the class proportions in each split mirror the overall proportions.  
This is critical for:
- Imbalanced classification (fraud, disease diagnosis, churn)
- Multi-class problems where some classes are rare
- Geographic or temporal subgroups

Let's compare the two approaches visually.

In [ ]:
# Run 50 random splits WITHOUT stratification and measure fraud rate per split
fraud_rates_no_strat  = {'train': [], 'test': []}
fraud_rates_stratified = {'train': [], 'test': []}

for seed in range(50):
    # Without stratification
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=seed)
    fraud_rates_no_strat['train'].append(ytr.mean() * 100)
    fraud_rates_no_strat['test'].append(yte.mean() * 100)

    # With stratification
    Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.20,
                                                stratify=y, random_state=seed)
    fraud_rates_stratified['train'].append(ytr2.mean() * 100)
    fraud_rates_stratified['test'].append(yte2.mean() * 100)

# ---- Plot ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, rates, title in zip(
    axes,
    [fraud_rates_no_strat, fraud_rates_stratified],
    ['WITHOUT stratify=y', 'WITH stratify=y']
):
    ax.plot(rates['train'], marker='o', markersize=4, label='Train fraud %', alpha=0.7)
    ax.plot(rates['test'],  marker='s', markersize=4, label='Test fraud %',  alpha=0.7)
    ax.axhline(y.mean()*100, color='red', linestyle='--', linewidth=1.5,
               label=f'True rate ({y.mean()*100:.2f} %)')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Random seed index')
    ax.set_ylabel('Fraud rate (%)')
    ax.legend(fontsize=9)

plt.suptitle('Fraud Rate Stability Across 50 Random Seeds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Without stratification — test fraud rate range:',
      f"{min(fraud_rates_no_strat['test']):.2f} % – {max(fraud_rates_no_strat['test']):.2f} %")
print('With stratification    — test fraud rate range:',
      f"{min(fraud_rates_stratified['test']):.2f} % – {max(fraud_rates_stratified['test']):.2f} %")

## Section 4: Random Seeds and Reproducibility

A `random_state` (or equivalently `np.random.seed()`) pins the pseudo-random number generator so that:
- Your teammate running the same notebook gets the **exact same split**.
- Your results are **reproducible** next month when you re-run the notebook.
- Your code review is **auditable** — reviewers can verify your exact experimental setup.

> Choosing `random_state=42` is a convention, not a magic number. 0, 7, 123 all work equally well.

**What changes across seeds?** The assignment of rows to splits changes, so the model sees different training examples — leading to different learned weights and different evaluation metrics.  
Cross-validation averages this out; a single holdout is subject to this variance.

In [ ]:
# Train the same logistic regression 30 times with different random seeds
# to visualise how much AUC varies due to split alone

seeds  = list(range(30))
aucs   = []

scaler = StandardScaler()

for seed in seeds:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                           stratify=y, random_state=seed)
    Xtr_sc = scaler.fit_transform(Xtr)
    Xte_sc = scaler.transform(Xte)
    model  = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0)
    model.fit(Xtr_sc, ytr)
    aucs.append(roc_auc_score(yte, model.predict_proba(Xte_sc)[:, 1]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(seeds, aucs, marker='o', color='steelblue', linewidth=1.5)
ax.axhline(np.mean(aucs), color='red', linestyle='--',
           label=f'Mean AUC = {np.mean(aucs):.4f}')
ax.fill_between(seeds,
                np.mean(aucs) - np.std(aucs),
                np.mean(aucs) + np.std(aucs),
                alpha=0.15, color='red', label=f'±1 std = {np.std(aucs):.4f}')
ax.set_xlabel('Random seed')
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC Variance Across 30 Different Train/Test Splits\n(same model, same data, different random seed)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'AUC range  : {min(aucs):.4f} – {max(aucs):.4f}')
print(f'AUC mean   : {np.mean(aucs):.4f}')
print(f'AUC std    : {np.std(aucs):.4f}')
print()
print('Lesson: a single split can produce a result that is up to',
      f'{(max(aucs)-min(aucs)):.4f} AUC points away from the true average.',
      'Use cross-validation for a stable estimate.')

## Section 5: Cross-Validation (k-Fold)

Cross-validation answers the question: **"How would this model perform on any unseen data, not just this particular split?"**

### How k-Fold CV works:

```
Full training data → split into k equal folds

Fold 1: [VAL] [TR ] [TR ] [TR ] [TR ]  → AUC_1
Fold 2: [TR ] [VAL] [TR ] [TR ] [TR ]  → AUC_2
Fold 3: [TR ] [TR ] [VAL] [TR ] [TR ]  → AUC_3
Fold 4: [TR ] [TR ] [TR ] [VAL] [TR ]  → AUC_4
Fold 5: [TR ] [TR ] [TR ] [TR ] [VAL]  → AUC_5

Final estimate: mean(AUC_1 … AUC_5) ± std
```

**StratifiedKFold** additionally ensures that each fold maintains the same class proportion as the full dataset — essential for imbalanced fraud data.

### When to use CV vs a single holdout:
- **Small dataset (< 5,000 rows):** use CV — you cannot afford to lock away 15-20 % as a static val set.
- **Large dataset (> 100,000 rows):** simple holdout is fine — it is fast and the large val set gives a stable estimate anyway.

In [ ]:
# 5-fold Stratified Cross-Validation on the TRAINING portion
# (We reserve X_test and never touch it here)

pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_aucs = []
fold_details = []

for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    Xf_tr, Xf_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    yf_tr, yf_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    pipe_cv.fit(Xf_tr, yf_tr)
    proba = pipe_cv.predict_proba(Xf_val)[:, 1]
    auc   = roc_auc_score(yf_val, proba)
    fold_aucs.append(auc)

    fold_details.append({
        'Fold'             : fold_idx,
        'Train size'       : len(yf_tr),
        'Val size'         : len(yf_val),
        'Fraud in train %' : f'{yf_tr.mean()*100:.2f}',
        'Fraud in val %'   : f'{yf_val.mean()*100:.2f}',
        'AUC-ROC'          : f'{auc:.4f}',
    })

df_folds = pd.DataFrame(fold_details)
print(df_folds.to_string(index=False))
print()
print(f'CV AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print()
print('Interpretation: the model achieves approximately',
      f'{np.mean(fold_aucs):.2f} AUC on unseen data,',
      f'with low variance (std = {np.std(fold_aucs):.4f}).')

In [ ]:
# Visualise fold-by-fold AUC
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2196F3' if a > np.mean(fold_aucs) else '#F44336' for a in fold_aucs]
bars = ax.bar([f'Fold {i}' for i in range(1, 6)], fold_aucs,
              color=colors, edgecolor='black', linewidth=0.7, alpha=0.85)
ax.axhline(np.mean(fold_aucs), linestyle='--', color='black',
           label=f'Mean = {np.mean(fold_aucs):.4f}')
ax.axhline(np.mean(fold_aucs) + np.std(fold_aucs), linestyle=':',
           color='grey', alpha=0.7)
ax.axhline(np.mean(fold_aucs) - np.std(fold_aucs), linestyle=':',
           color='grey', alpha=0.7, label=f'±1 std = {np.std(fold_aucs):.4f}')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('AUC-ROC')
ax.set_title('5-Fold Stratified CV — Fold-by-Fold AUC\n(Logistic Regression, Fraud Detection)', fontsize=12)
ax.legend()
for bar, val in zip(bars, fold_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Section 6: Temporal Splits — Respecting Time Order

Fraud patterns shift over time. Fraudsters adapt to detection systems.  
If you randomly split a year of transactions, the model might learn from December data to predict January fraud — **information from the future leaks into the past**.

### The correct approach for time-series data:

```
Timeline:  Jan ──────────────────────────── Dec

WRONG (random split):  [Test] [Train] [Test] [Train] [Train]
RIGHT (temporal split): [────── Train ──────] [── Test ──]
```

All training data must come **before** all test data.

### TimeSeriesSplit — Expanding Window

Instead of a single cut point, `TimeSeriesSplit` creates multiple expanding train windows with test windows that advance forward in time. This is more realistic: as more data arrives, the model is retrained and evaluated on future windows.

In [ ]:
# ---- 1. Simple temporal split: first 80% of time → train, last 20% → test ----
df_sorted = df.sort_values('timestamp').reset_index(drop=True)
X_sorted  = df_sorted[FEATURES]
y_sorted  = df_sorted[TARGET]

cutoff_idx     = int(len(df_sorted) * 0.80)
cutoff_date    = df_sorted['timestamp'].iloc[cutoff_idx]

X_temp_train   = X_sorted.iloc[:cutoff_idx]
y_temp_train   = y_sorted.iloc[:cutoff_idx]
X_temp_test    = X_sorted.iloc[cutoff_idx:]
y_temp_test    = y_sorted.iloc[cutoff_idx:]

print(f'Temporal split cutoff date : {cutoff_date.date()}')
print(f'Train: {len(X_temp_train):,} rows | Fraud {y_temp_train.mean()*100:.2f} %')
print(f'Test : {len(X_temp_test):,}  rows | Fraud {y_temp_test.mean()*100:.2f} %')
print()

# Compare random split vs temporal split AUC
pipe = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
])

# Random split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.20,
                                                stratify=y, random_state=RANDOM_STATE)
pipe.fit(Xr_tr, yr_tr)
auc_random = roc_auc_score(yr_te, pipe.predict_proba(Xr_te)[:, 1])

# Temporal split
pipe.fit(X_temp_train, y_temp_train)
auc_temporal = roc_auc_score(y_temp_test, pipe.predict_proba(X_temp_test)[:, 1])

print(f'AUC — Random split   : {auc_random:.4f}')
print(f'AUC — Temporal split : {auc_temporal:.4f}')
print()
print('The temporal AUC is typically lower — it is a harder,',
      'more realistic evaluation: the model must generalise forward in time.')

In [ ]:
# ---- 2. TimeSeriesSplit: visualise expanding windows ----------------------
tss    = TimeSeriesSplit(n_splits=5)
X_ts   = X_sorted.values
y_ts   = y_sorted.values

fig, ax = plt.subplots(figsize=(12, 5))

colors_train = plt.cm.Blues(np.linspace(0.35, 0.75, 5))
colors_test  = plt.cm.Reds(np.linspace(0.35, 0.75, 5))

ts_aucs = []
for fold_i, (tr_idx, te_idx) in enumerate(tss.split(X_ts)):
    # Plot
    ax.barh(y=fold_i, width=len(tr_idx), left=0,
            height=0.5, color=colors_train[fold_i],
            label='Train' if fold_i == 0 else '')
    ax.barh(y=fold_i, width=len(te_idx), left=len(tr_idx),
            height=0.5, color=colors_test[fold_i],
            label='Test' if fold_i == 0 else '')
    # AUC
    pipe.fit(X_ts[tr_idx], y_ts[tr_idx])
    auc_ts = roc_auc_score(y_ts[te_idx], pipe.predict_proba(X_ts[te_idx])[:, 1])
    ts_aucs.append(auc_ts)
    ax.text(len(tr_idx) + len(te_idx) + 20, fold_i,
            f'AUC={auc_ts:.3f}', va='center', fontsize=9)

ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Transaction index (sorted by time)')
ax.set_title('TimeSeriesSplit — Expanding Training Window\n(each fold adds more history, tests on the next window)', fontsize=12)
train_patch = mpatches.Patch(color=colors_train[2], label='Train')
test_patch  = mpatches.Patch(color=colors_test[2],  label='Test')
ax.legend(handles=[train_patch, test_patch], loc='lower right')
plt.tight_layout()
plt.show()

print(f'TimeSeriesSplit AUC per fold : {[f"{a:.4f}" for a in ts_aucs]}')
print(f'Mean ± std                   : {np.mean(ts_aucs):.4f} ± {np.std(ts_aucs):.4f}')

## Section 7: Group Splits — When Rows Are Not Independent

Our dataset has **1,000 users**, each with ~10 transactions.  
Transactions from the same user are correlated — they share spending habits, device type, location, etc.

**The problem with ordinary random splits:**  
User 42 might have 8 transactions in the training set and 2 in the test set.  
The model can memorise user-specific patterns (average spend, typical merchant categories) and "cheat" on the test set — producing overoptimistic AUC.

**The solution: `GroupKFold`**  
Every transaction of user 42 goes into either the train fold **or** the test fold, never both.  
This tests whether the model generalises to **new users it has never seen** — which is what you actually care about in production.

### Rule of thumb for group splits:
- **user_id in both sets?** → group leakage — use GroupKFold.
- **device_id in both sets?** → same issue.
- **store_id in both sets?** → same issue.

In [ ]:
groups = df['user_id'].values

gkf = GroupKFold(n_splits=5)

fold_stats = []
group_aucs = []

for fold_i, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    Xg_tr, Xg_te = X.iloc[tr_idx], X.iloc[te_idx]
    yg_tr, yg_te = y.iloc[tr_idx], y.iloc[te_idx]

    train_users = set(groups[tr_idx])
    test_users  = set(groups[te_idx])
    overlap     = train_users & test_users  # should be empty

    pipe.fit(Xg_tr, yg_tr)
    auc_g = roc_auc_score(yg_te, pipe.predict_proba(Xg_te)[:, 1])
    group_aucs.append(auc_g)

    fold_stats.append({
        'Fold'             : fold_i,
        'Train rows'       : len(tr_idx),
        'Test rows'        : len(te_idx),
        'Train users'      : len(train_users),
        'Test users'       : len(test_users),
        'User overlap'     : len(overlap),
        'AUC'              : f'{auc_g:.4f}',
    })

print(pd.DataFrame(fold_stats).to_string(index=False))
print()
print('Key observation: User overlap = 0 in every fold.')
print('The model is tested on entirely new users it has never seen during training.')
print(f'GroupKFold CV AUC: {np.mean(group_aucs):.4f} ± {np.std(group_aucs):.4f}')
print()
print(f'Standard StratifiedKFold AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print('(GroupKFold AUC is typically lower — it is a harder test.)')

## Section 8: Holdout vs Cross-Validation — When to Use Which

Both approaches estimate how well a model will generalise. The difference is **how much data they use** and **how stable the estimate is**.

| Criterion | Single holdout | k-Fold CV |
|---|---|---|
| Dataset size | Works well ≥ 50,000 rows | Preferred < 10,000 rows |
| Variance of estimate | High (depends on which rows ended up where) | Low (averages over k splits) |
| Compute cost | Cheap: train once | k× more expensive |
| Reproducibility | Simple, one split | Need to fix shuffle seed |
| Suitable for time series | Only with temporal cut | Only with TimeSeriesSplit |

Let's make the variance comparison concrete.

In [ ]:
# Compare: single holdout (25 % test) vs 5-fold CV
# Run each 30 times to see distribution of AUC estimates

holdout_aucs = []
cv5_means    = []

for seed in range(30):
    Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X, y, test_size=0.25,
                                                    stratify=y, random_state=seed)
    pipe.fit(Xh_tr, yh_tr)
    holdout_aucs.append(roc_auc_score(yh_te, pipe.predict_proba(Xh_te)[:, 1]))

    cv_scores = cross_val_score(pipe, X, y,
                                cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                                scoring='roc_auc')
    cv5_means.append(cv_scores.mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, label, color in zip(
    axes,
    [holdout_aucs, cv5_means],
    ['Single Holdout (25 %)', '5-Fold CV Mean'],
    ['#FF7043', '#42A5F5']
):
    ax.hist(data, bins=12, color=color, edgecolor='black', alpha=0.8)
    ax.axvline(np.mean(data), color='black', linestyle='--',
               label=f'Mean = {np.mean(data):.4f}')
    ax.set_title(f'{label}\nstd = {np.std(data):.4f}', fontsize=11)
    ax.set_xlabel('AUC-ROC')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Distribution of AUC Estimates: Holdout vs 5-Fold CV (30 repetitions)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Holdout std : {np.std(holdout_aucs):.4f}  (wider spread — less stable)')
print(f'CV-5 std    : {np.std(cv5_means):.4f}  (tighter spread — more stable)')
print(f'Holdout mean: {np.mean(holdout_aucs):.4f}')
print(f'CV-5 mean   : {np.mean(cv5_means):.4f}')

## Section 9: Summary — Decision Guide

Use this flowchart when deciding which splitting strategy to use:

```
Is your data ordered by time?
    YES → Use TimeSeriesSplit (or a single temporal cut)
    NO  →
        Are rows correlated by group (user_id, store_id, device_id)?
            YES → Use GroupKFold
            NO  →
                Is your dataset small (< 5,000 rows)?
                    YES → Use StratifiedKFold (cross-validation)
                    NO  → Use stratified train_test_split
                             (three-way if tuning hyperparameters)
```

**Always:**
- Set `stratify=y` when you have class imbalance.
- Set `random_state` for reproducibility.
- Touch the test set exactly once, after all decisions are finalised.
- Never pre-process data before splitting — preprocessing must be fit on training data only.

## Self-Check Questions

Answer these questions on your own, then expand the details to check.

---

**Q1. You have 500 training examples. Should you use a holdout validation set or cross-validation? Why?**

<details>
<summary>Show answer</summary>

**Cross-validation.** With only 500 examples, a 20 % holdout leaves 400 rows for training and 100 for validation. That validation set is too small to give a stable estimate of model performance — a single unlucky assignment of rows can swing your AUC by 0.05 or more. 5-fold CV trains on 400 examples each time but averages over 5 different validation sets (each of 100), giving a much more reliable estimate of generalisation performance with the same amount of data.

</details>

---

**Q2. You tune 20 hyperparameter combinations using your test set to pick the best. Why is your final reported accuracy likely overly optimistic?**

<details>
<summary>Show answer</summary>

When you compare 20 models on the test set and report the performance of the best one, you are exploiting random fluctuations in test-set performance. By chance, some configuration will score well on these specific test examples — that does not mean it is genuinely better. You have essentially trained on the test set by using it to make a decision. The reported accuracy is inflated because it benefited from this selection bias. The fix: use the test set only once, after all hyperparameter decisions are made using a validation set or cross-validation.

</details>

---

**Q3. You are predicting next month's fraud patterns using historical transaction data. Why is StratifiedKFold wrong here?**

<details>
<summary>Show answer</summary>

StratifiedKFold shuffles the data before splitting. This means some future transactions end up in the training fold while some past transactions end up in the validation fold. The model can implicitly learn from future information (future fraud patterns, seasonal trends, emerging fraud tactics) when predicting the past, which is impossible in a real deployment scenario. The correct approach is `TimeSeriesSplit`, which ensures that every training fold contains only data from before the validation window.

</details>

---

**Q4. Without `stratify=y`, you split a 99:1 imbalanced dataset. What could go wrong?**

<details>
<summary>Show answer</summary>

A 99:1 imbalance means only 1 % of rows are the positive class. In a random 80/20 split of 1,000 rows, you expect about 10 positives in the test set. Pure chance can easily give you 0, 3, or 20 positives in the test set. If 0 positives land in the test set, metrics like recall and F1 become undefined. Even if a few land in the test set, the measured performance is extremely noisy. Stratification guarantees that the 1 % rate is preserved in both splits, giving a consistent and meaningful evaluation.

</details>